# 03 · Logistic regression: CV tuning, then one refit on the test split

LR sees the flattened ImageNet-normalised 224 × 224 patch (150,528 features), plus the two
z-scored morphometrics for `lr_morph`, scaled with `StandardScaler`, L-BFGS, balanced class
weights. The inverse regularisation strength C (1 … 10⁻⁶) and, for three-class, an extra
weight on Perfect (1, 2, 4) were chosen by **3-fold cross-validation inside the training
split, whole animals held out per fold**. The test split is scored once, with the choice.

Selected for the manuscript: **C = 10⁻⁶ for every task; Perfect weight 4**.

Run times: feature cache ~10 min, CV ≈ 12 h (three-class fits at large C take ~30 min each), refit
~5 min per model. With `RUN = False` the notebook shows the reference CV table and the
choice it implies, and reads the refit results.

**`RUN`** decides what this notebook does.
* With the original data on disk, `False` (default) reads the results of the run behind
  the manuscript and checks them; `True` re-trains into `runs/rerun`.
* On the synthetic data, `RUN` is `True`: there are no saved results to read, so the models
  are trained (into `runs/synthetic/rerun`; finished runs are skipped).

The manuscript's results folders are never overwritten, and nothing is written into the
dataset.

In [ ]:
import json, os, subprocess, sys, warnings
from pathlib import Path

# Find the repository folder (it holds scripts/paths.py) from wherever Jupyter was started.
HERE = Path.cwd()
while not (HERE / "scripts" / "paths.py").is_file() and HERE != HERE.parent:
    HERE = HERE.parent
sys.path.insert(0, str(HERE / "scripts"))
warnings.filterwarnings("ignore", category=FutureWarning)
# The histology images are not distributed. Without them this notebook runs on the
# synthetic dataset (scripts/make_synthetic_data.py) and writes to runs/synthetic/.
SYNTHETIC = (os.environ.get("VML_SYNTHETIC") == "1"          # forced, or no original data
             or not ((HERE / "data" / "Dataset_balanced" / "dataset.yaml").is_file() or "VML_DATA" in os.environ))
if SYNTHETIC:
    os.environ["VML_SYNTHETIC"] = "1"
import numpy as np
import pandas as pd
import paths as P
pd.set_option("display.width", 200, "display.max_columns", 30)
PY = sys.executable
if SYNTHETIC:   # generate and balance the synthetic data once (both steps skip existing output)
    subprocess.run([PY, str(HERE / "scripts" / "make_synthetic_data.py"), "--out", str(P.DATASETS)], check=True)
    subprocess.run([PY, str(HERE / "scripts" / "build_dataset.py"), "--only", "Dataset_balanced"],
                   check=True, capture_output=True)
print("mode           :", "SYNTHETIC data (not the manuscript's)" if SYNTHETIC else "original data")
print("dataset        :", P.DATA)
print("human data     :", P.HUMAN_ROOT)
print("results (used) :", P.RESULTS)

In [ ]:
RUN = SYNTHETIC                           # <- True: train into RERUN_BASE (always on synthetic data)
RERUN_BASE = P.WORK / "rerun"
BASE = RERUN_BASE if RUN else P.RESULTS   # where this notebook reads results from

In [ ]:
import lr_tuning
cv_csv = lr_tuning.OUT if RUN else P.REFERENCE / "lr_tuning_cv.csv"
if RUN:
    lr_tuning.extract()      # -> runs/feature_cache (train split only)
    lr_tuning.cv()           # -> runs/lr_tuning_cv.csv (resumable)
cvt = pd.read_csv(cv_csv)
cvt[["task", "variant", "C", "w", "patch_macro", "patch_f1_0", "wsi_macro", "wsi_f1_0", "minutes"]]

In [ ]:
pick = lr_tuning.choose(cv_csv)
chosen = pd.DataFrame([dict(task=t, model=v, C=r["C"], perfect_weight=r["w"], cv_patch_macro=r["patch_macro"],
                            baseline_cv_patch_macro=r["baseline_patch_macro"]) for (t, v), r in pick.items()])
if not SYNTHETIC:   # the manuscript's choice; on synthetic data the CV picks its own
    ref = json.loads((P.REFERENCE / "lr_tuned_params.json").read_text())
    chosen["matches_manuscript"] = [(ref[f"{r.model}__{r.task}"]["C"], ref[f"{r.model}__{r.task}"]["w"]) == (r.C, r.perfect_weight)
                                    for r in chosen.itertuples()]
chosen

In [ ]:
RES = P.roots(BASE)["lr"]
if RUN:
    lr_tuning.final(RES, cv_csv)
from manuscript_numbers import compute
N = compute(BASE, models=["lr", "lr_morph"], human=[])
pd.DataFrame(N["rat"]).T[["patch_f1_pos", "wsi_f1_pos", "patch_macro", "wsi_macro", "auc"]].round(3)